In [0]:
%sql
WITH viacep AS (
    SELECT
        cep,
        get_json_object(json_original, '$.logradouro') AS logradouro,
        get_json_object(json_original, '$.bairro') AS bairro,
        get_json_object(json_original, '$.localidade') AS cidade,
        get_json_object(json_original, '$.uf') AS uf
    FROM geocep.default.ref_viacep
    WHERE status = 'OK'
),
brasilapi AS (
    SELECT
        cep,
        get_json_object(json_original, '$.street') AS logradouro,
        get_json_object(json_original, '$.neighborhood') AS bairro,
        get_json_object(json_original, '$.city') AS cidade,
        get_json_object(json_original, '$.state') AS uf
    FROM geocep.default.ref_brasilapi
    WHERE status = 'OK'
)
SELECT
    COALESCE(v.cep, b.cep) AS cep,
    COALESCE(NULLIF(v.logradouro, ''), b.logradouro) AS logradouro,
    COALESCE(NULLIF(v.bairro, ''), b.bairro) AS bairro,
    COALESCE(NULLIF(v.cidade, ''), b.cidade) AS cidade,
    COALESCE(NULLIF(v.uf, ''), b.uf) AS uf,

    CASE
        WHEN v.cep IS NULL THEN 'FALLBACK_BRASILAPI'
        WHEN b.cep IS NULL THEN 'SOMENTE_VIACEP'
        WHEN LOWER(TRIM(v.logradouro)) =
             LOWER(TRIM(b.logradouro)) THEN 'VALIDADO'
        ELSE 'DIVERGENCIA'
    END AS status_qualidade

FROM viacep v
FULL OUTER JOIN brasilapi b
    ON v.cep = b.cep;